# Training YOLOv8 Hook — 100 Epoch

> Catatan: model ini mendeteksi hook (bounding box). Posisi X/Y global ROV tetap membutuhkan kalibrasi kamera, geometri/map hook, dan tahap pose/localization terpisah.

In [ ]:
%pip -q install ultralytics

import os
import shutil
import zipfile
from pathlib import Path

import torch
from ultralytics import YOLO

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('Torch:', torch.__version__)
print('Device:', DEVICE)
if DEVICE == 'cpu':
    print('PERINGATAN: GPU Colab belum aktif. Runtime > Change runtime type > T4 GPU.')

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
assert zip_names, 'Upload file ZIP dataset terlebih dahulu.'
ZIP_PATH = Path('/content') / zip_names[0]
RAW_ROOT = Path('/content/hook_dataset_raw')
if RAW_ROOT.exists():
    shutil.rmtree(RAW_ROOT)
RAW_ROOT.mkdir(parents=True)
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(RAW_ROOT)
print('Dataset diekstrak ke:', RAW_ROOT)
print('Ukuran ZIP (MB):', round(ZIP_PATH.stat().st_size / 1024**2, 1))

In [ ]:
# Cari root yang benar dan tulis YAML baru dengan path yang valid untuk Colab.
train_images = next(RAW_ROOT.rglob('train/images'), None)
valid_images = next(RAW_ROOT.rglob('valid/images'), None)
test_images = next(RAW_ROOT.rglob('test/images'), None)
assert train_images and valid_images and test_images, 'Struktur train/valid/test tidak ditemukan.'
DATASET_ROOT = train_images.parent.parent
DATA_YAML = Path('/content/hook_data.yaml')
DATA_YAML.write_text(
    f"path: {DATASET_ROOT}\n"
    "train: train/images\n"
    "val: valid/images\n"
    "test: test/images\n"
    "names:\n  0: Hook\n"
)
print(DATA_YAML.read_text())

In [ ]:
# Validasi ringan label YOLO. Polygon > 5 field diubah menjadi bbox detection.
split_dirs = {
    'train': DATASET_ROOT / 'train',
    'valid': DATASET_ROOT / 'valid',
    'test': DATASET_ROOT / 'test',
}
converted = 0
bad = []
counts = {}
for split, split_dir in split_dirs.items():
    label_dir = split_dir / 'labels'
    rows = empty = 0
    for label_path in label_dir.glob('*.txt'):
        text = label_path.read_text().strip()
        if not text:
            empty += 1
            continue
        output = []
        changed = False
        for line_no, line in enumerate(text.splitlines(), 1):
            values = line.split()
            try:
                numbers = [float(v) for v in values]
            except ValueError:
                bad.append((str(label_path), line_no, line))
                continue
            if len(numbers) == 5:
                output.append(' '.join(values))
            elif len(numbers) > 5 and (len(numbers) - 1) % 2 == 0:
                cls = numbers[0]
                xs = numbers[1::2]
                ys = numbers[2::2]
                x_min, x_max = max(0.0, min(xs)), min(1.0, max(xs))
                y_min, y_max = max(0.0, min(ys)), min(1.0, max(ys))
                output.append(f'{cls:g} {(x_min+x_max)/2:g} {(y_min+y_max)/2:g} {x_max-x_min:g} {y_max-y_min:g}')
                changed = True
                converted += 1
            else:
                bad.append((str(label_path), line_no, line))
        if changed:
            label_path.write_text('\n'.join(output) + '\n')
        rows += len(output)
    counts[split] = {'images': len(list((split_dir / 'images').glob('*'))), 'label_files': len(list(label_dir.glob('*.txt'))), 'rows': rows, 'empty': empty}
assert not bad, f'Label invalid: {bad[:3]}'
print(counts)
print('Polygon rows converted to bbox:', converted)

## Training

Target training ditetapkan 100 epoch. `batch=-1` membiarkan Ultralytics memilih batch sesuai VRAM Colab.

In [ ]:
model = YOLO('yolov8n.pt')
results = model.train(
    data=str(DATA_YAML),
    epochs=100, # Jumlah epoch untuk pelatihan
    imgsz=640, # Ukuran gambar input
    batch=-1, # Jumlah batch (otomatis)
    device=DEVICE, # Perangkat untuk pelatihan (GPU/CPU)
    workers=6, # Jumlah worker untuk dataloader
    patience=30, # Jumlah epoch tanpa peningkatan sebelum berhentiS
    project='/content/hook_yolo_runs',
    name='yolov8n_hook_100',
    exist_ok=True,
    pretrained=True,
    plots=True,
)
BEST_PT = Path('/content/hook_yolo_runs/yolov8n_hook_100/weights/best.pt')
assert BEST_PT.exists(), f'Checkpoint tidak ditemukan: {BEST_PT}'
print('Best model:', BEST_PT)

In [ ]:
# Evaluasi test: hasil ini masih preliminary karena split berasal dari frame video yang sama.
best_model = YOLO(str(BEST_PT))
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=640,
    batch=-1,
    device=DEVICE,
    project='/content/hook_yolo_runs',
    name='yolov8n_hook_100_test',
    exist_ok=True,
    plots=True,
)
print(test_metrics.results_dict)

In [ ]:
# Contoh inferensi pada beberapa frame underwater dari test split.
test_sources = sorted((DATASET_ROOT / 'test/images').glob('*.jpg'))[:12]
predictions = best_model.predict(
    source=[str(p) for p in test_sources],
    imgsz=640,
    conf=0.25,
    device=DEVICE,
    save=True,
    project='/content/hook_yolo_runs',
    name='yolov8n_hook_100_preview',
    exist_ok=True,
)
print('Preview:', '/content/hook_yolo_runs/yolov8n_hook_100_preview')

In [ ]:
# Download model hasil training untuk dipakai pada pipeline laptop.
from google.colab import files
files.download(str(BEST_PT))

## Batasan hasil

- `best.pt` baru menyelesaikan deteksi hook, belum menghasilkan X/Y global ROV.
- Untuk bantuan X/Y: gunakan pusat bbox sebagai error piksel relatif terlebih dahulu; tahap berikutnya perlu kalibrasi kamera, ukuran/geometri hook, dan transformasi ke frame arena.
- Dataset saat ini berasal dari satu sequence video dengan split frame. Untuk angka final, tambahkan video underwater baru dan buat split berdasarkan sequence, bukan frame acak.
- Jangan hubungkan output ini langsung ke perintah gerak sebelum threshold confidence, timeout/stale detection, dan pengujian kolam disetujui.